# Keras/CNN: ????????????? ????? ????????? ???????

???? ??????? ???????? ????????????? ??????????? ????????? ???????. ?????? ??????? ?? ???????????? ? ??????????? ???????? `verified_materials/mobile_robot_dataset`, ??? ??????????? ????????? ?? ?????? `train`, `validation`, `test` ? ??????? `legged_robot`, `tracked_robot`, `wheeled_robot`.


In [ ]:
import os
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

cwd = Path.cwd()
MATERIALS = cwd / "verified_materials" if (cwd / "verified_materials").exists() else cwd
DATASET = MATERIALS / "mobile_robot_dataset"
OUTPUTS = MATERIALS / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 128
BATCH_SIZE = 12
SEED = 42

tf.keras.utils.set_random_seed(SEED)

if not DATASET.exists():
    raise FileNotFoundError(f"Dataset not found: {DATASET}")

print("TensorFlow:", tf.__version__)
print("Dataset:", DATASET)


In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET / "train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
    seed=SEED,
)

validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET / "validation",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET / "test",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names = train_ds.class_names
class_titles = {
    "legged_robot": "????????",
    "tracked_robot": "??????????",
    "wheeled_robot": "????????",
}

for split in ["train", "validation", "test"]:
    split_count = sum(1 for _ in (DATASET / split).glob("*/*.jpg"))
    print(f"{split}: {split_count} images")
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(256, seed=SEED).prefetch(AUTOTUNE)
validation_ds = validation_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


In [ ]:
sample_images, sample_labels = next(iter(train_ds.unbatch().batch(18)))

fig, axes = plt.subplots(3, 6, figsize=(10, 5))
for ax, image, label in zip(axes.ravel(), sample_images, sample_labels):
    class_name = class_names[int(label)]
    ax.imshow(image.numpy().astype("uint8"))
    ax.axis("off")
    ax.set_title(class_titles.get(class_name, class_name), fontsize=9)
fig.tight_layout()
fig.savefig(OUTPUTS / "keras_robot_training_examples.png", dpi=160)
plt.show()


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu"),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(len(class_names), activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
history = model.fit(
    train_ds,
    epochs=12,
    validation_data=validation_ds,
    verbose=2,
)

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy on mobile robot dataset: {test_accuracy:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history["accuracy"], marker="o", label="train accuracy")
ax.plot(history.history["val_accuracy"], marker="o", label="validation accuracy")
ax.set_xlabel("?????")
ax.set_ylabel("Accuracy")
ax.set_title("???????? ????????????? ????????? ???????")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUTS / "keras_robot_history.png", dpi=160)
plt.show()


In [ ]:
test_images, test_labels = next(iter(test_ds.unbatch().batch(9)))
probs = model.predict(test_images, verbose=0)
pred_labels = np.argmax(probs, axis=1)

fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for ax, image, true_label, pred_label, prob in zip(axes.ravel(), test_images, test_labels, pred_labels, probs):
    true_name = class_names[int(true_label)]
    pred_name = class_names[int(pred_label)]
    color = "green" if int(true_label) == int(pred_label) else "red"
    ax.imshow(image.numpy().astype("uint8"))
    ax.axis("off")
    ax.set_title(
        f"true: {class_titles.get(true_name, true_name)}
"
        f"pred: {class_titles.get(pred_name, pred_name)} ({prob[pred_label]:.2f})",
        color=color,
        fontsize=9,
    )
fig.tight_layout()
fig.savefig(OUTPUTS / "keras_robot_predictions.png", dpi=160)
plt.show()
